# TruthLens image detector on a Colab or Kaggle GPU

Trains the image detector on real GenImage images: ADM fakes and their ImageNet originals for training, validation and test, and BigGAN fakes that training never sees, to measure how well it carries over to a new generator. Then it evaluates, exports ONNX and packs the results.

## Setup

Use a T4 GPU runtime (Runtime, Change runtime type). Add your Kaggle API token as a Colab secret named `KAGGLE_API_TOKEN` (the key icon on the left) and allow this notebook to read it. Everything that must survive a disconnect goes to Google Drive.

On Kaggle, attach the datasets `vtphatt2/genimage-adm` and `vtphatt2/genimage-biggan`, turn on Internet and a GPU, and use Save & Run All. The subset is linked from the attached datasets, so nothing is downloaded and no token is needed.

In [ ]:
REPO = 'https://github.com/Shashank17singh/TruthLens.git'
BRANCH = 'main'
import os

# Kaggle's "Save & Run All" runs headless for up to 12 hours with no idle timeout; Colab free stops after about 90 idle minutes.
KAGGLE = os.path.exists('/kaggle/working')
HOME = '/tmp/TruthLens' if KAGGLE else '/content/TruthLens'  # /tmp on Kaggle: /kaggle/working is capped at 20 GB
# Training stops cleanly after the epoch that would overrun this many minutes; rerunning it resumes.
BUDGET = 300 if KAGGLE else 75
if not os.path.exists(HOME):
    !git clone -q --branch {BRANCH} {REPO} {HOME}
%cd {HOME}
!git pull -q
assert os.path.exists('ml/image/train.py'), 'This branch does not have ml/image/train.py. Push your commits, then run this cell again.'

In [ ]:
!pip install -q -r ml/requirements.txt
import torch

GPU = torch.cuda.is_available()
print(torch.cuda.get_device_name(0) if GPU else 'No GPU. Runtime, Change runtime type, T4 GPU.')

If pip asks you to restart the session, restart it and run the two cells above again. The clone is kept.

In [ ]:
if KAGGLE:
    DRIVE = '/kaggle/working/truthlens'  # saved as the notebook's output; download it from the Output tab
else:
    from google.colab import drive, userdata

    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/truthlens'
    os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
os.makedirs(DRIVE, exist_ok=True)

## Data

The Kaggle copies of GenImage are 39 GB (ADM) and 24 GB (BigGAN), more than a free Colab disk holds once unpacked. So each archive is downloaded, a seeded random subset is copied straight out of it, and the archive is deleted. The subset is saved to Drive, so after a reset the next cell restores it instead of downloading again.

In [ ]:
import glob
import random
import shutil
import subprocess
import tarfile
import zipfile

RAW = 'data/raw/genimage'
ZIPS = '/content/zips'
CACHE = f'{DRIVE}/genimage_subset.tar'
PER_CLASS = 20000      # ADM training-set images per class (fake and real)
UNSEEN_FAKES = 4000    # BigGAN fakes, used only as the unseen-generator test
IMAGE_SUFFIXES = ('.png', '.jpg', '.jpeg', '.webp')


def link_mounted(generator, split_dir, classes, count):
    """Kaggle: link a seeded subset from the attached, already extracted dataset."""
    for cls in classes:
        folders = [d for depth in range(2, 7) for d in glob.glob('/kaggle/input/' + '*/' * depth + f'{split_dir}/{cls}')
                   if f'/{generator}/' in d.lower()]
        assert folders, f'{generator} {split_dir}/{cls} not found under /kaggle/input: attach the dataset'
        pool = sorted(os.path.join(folders[0], n) for n in os.listdir(folders[0]) if n.lower().endswith(IMAGE_SUFFIXES))
        for path in random.Random(1337).sample(pool, min(count, len(pool))):
            parts = path.split('/')
            # Same layout as the zip branch: genimage/<generator>/.../<cls>/<file>.
            target = os.path.join(RAW, *parts[[p.lower() for p in parts].index(generator):])
            os.makedirs(os.path.dirname(target), exist_ok=True)
            if not os.path.exists(target):
                os.symlink(path, target)
        print(generator, cls, min(count, len(pool)), 'of', len(pool))


def take(slug, generator, split_dir, classes, count):
    """Download one GenImage generator, copy a random subset out of the zip, delete the zip."""
    os.makedirs(ZIPS, exist_ok=True)
    print(shutil.disk_usage('/content').free // 2**30, 'GB free')
    subprocess.run(['kaggle', 'datasets', 'download', '-d', slug, '-p', ZIPS, '-q'], check=True)
    archive = glob.glob(f'{ZIPS}/*.zip')[0]
    with zipfile.ZipFile(archive) as bundle:
        names = [n for n in bundle.namelist() if n.lower().endswith(IMAGE_SUFFIXES)]
        print(len(names), 'images in', slug, '- first:', names[0])
        for cls in classes:
            pool = sorted(n for n in names if f'/{split_dir}/{cls}/' in n)
            assert pool, f'no {split_dir}/{cls}/ images in {slug}: check the layout printed above'
            for name in random.Random(1337).sample(pool, min(count, len(pool))):
                parts = name.split('/')
                lowered = [p.lower() for p in parts]
                assert generator in lowered, f'{name} has no {generator} folder'
                # Keep the path from the generator folder down, so the adapter sees genimage/<generator>/.../<cls>/<file>.
                target = os.path.join(RAW, *parts[lowered.index(generator):])
                os.makedirs(os.path.dirname(target), exist_ok=True)
                with bundle.open(name) as src, open(target, 'wb') as dst:
                    shutil.copyfileobj(src, dst)
            print(cls, min(count, len(pool)), 'of', len(pool))
    os.remove(archive)


if KAGGLE:
    link_mounted('adm', 'train', ['ai', 'nature'], PER_CLASS)
    link_mounted('biggan', 'val', ['ai'], UNSEEN_FAKES)
elif os.path.exists(CACHE):
    print('Restoring the subset from Drive')
    with tarfile.open(CACHE) as bundle:
        bundle.extractall('.', filter='data')
else:
    take('vtphatt2/genimage-adm', 'adm', 'train', ['ai', 'nature'], PER_CLASS)
    take('vtphatt2/genimage-biggan', 'biggan', 'val', ['ai'], UNSEEN_FAKES)
    with tarfile.open(CACHE, 'w') as bundle:
        bundle.add(RAW)
for generator in sorted(os.listdir(RAW)):
    for cls in ('ai', 'nature'):
        found = glob.glob(f'{RAW}/{generator}/**/{cls}/*', recursive=True)
        print(generator, cls, len(found))

In [ ]:
# Resizes every image and writes data/manifests/genimage.csv with grouped 70/15/15 splits.
# BigGAN is held out in ml/common/datasets.yaml, so its fakes go to unseen_test only.
!python -m ml.common.prepare genimage --workers 2

## Train

ConvNeXt-Tiny with the frequency branch, the default architecture. Each epoch is capped at 1,500 steps so one fits in a session, and running the cell again resumes from the last finished epoch on Drive.

In [ ]:
CKPT = f'{DRIVE}/checkpoints/image'
assert GPU, 'Training needs the T4 GPU runtime.'
!python -m ml.image.train model=convnext_tiny use_frequency=true manifest=data/manifests/genimage.csv checkpoint_dir={CKPT} tracker=none max_minutes={BUDGET} num_workers=2 steps_per_epoch=1500

## Evaluate and export

The report scores the held-out ADM test split and, separately, the BigGAN fakes against the real test images. Calibration and the threshold come from validation only.

In [ ]:
BEST = sorted(glob.glob(f'{CKPT}/*/*-best.pt'))[-1]
RUN = os.path.basename(os.path.dirname(BEST))
!python -m ml.image.evaluate {BEST} --output {DRIVE}/results/image/{RUN}
!python -m ml.image.export {BEST} --output-dir {DRIVE}/onnx

In [ ]:
import json

report = json.load(open(f'{DRIVE}/results/image/{RUN}/metrics.json'))
print(json.dumps({k: report.get(k) for k in ('test', 'unseen')}, indent=1))

## Hand back the results

This packs the metrics, figures, ONNX export, manifest balance and data card into one zip on Drive. Download it and send it over to wire the model into the app.

In [ ]:
PACK = f'{DRIVE}/truthlens_image_results'
shutil.rmtree(PACK, ignore_errors=True)
shutil.copytree(f'{DRIVE}/results/image/{RUN}', f'{PACK}/results/image/{RUN}')
for path in glob.glob(f'{DRIVE}/onnx/{RUN}*'):
    shutil.copy(path, f'{PACK}/')
shutil.copy('data/manifests/genimage.balance.json', f'{PACK}/')
if os.path.exists('docs/datacards/genimage.md'):
    shutil.copy('docs/datacards/genimage.md', f'{PACK}/')
shutil.make_archive(PACK, 'zip', PACK)
print('Download', PACK + '.zip')